# Required Library

In [32]:
# Libraries
import pandas as pd
input_folder_path = "DataSource/New Input/"
output_folder_path = "DataSource/Output/"

#### Game Title Process Method

In [13]:
def title_process (df):
    # remove everything in front of last column
    df['Game Title'] = df['Game Title'].str.replace(' : ', ':', regex = False).str.rsplit(':', n=1).str[-1].str.upper()

    # remove platform
    # pattern = '|'.join(platforms)
    # df['Game Title'] = df['Game Title'].str.replace(rf'\s*\b({pattern})\b.*', '', regex=True)
    
    return df

#### Item Code Process Method

In [14]:
def item_process (df):
    df['Item'] = df['Item'].apply(lambda x: '-'.join(x.split('-')[:2]))
    return df

# Sales

In [10]:
import pandas as pd
from bs4 import BeautifulSoup
import re

file_path = 'DataSource/New Input/SalesbyItembyLicensor-ForRoyaltyCal(Acc)new_Jan2025March2025_0707.xls'

print("Parsing XML... ")
with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
    soup = BeautifulSoup(f.read(), 'xml')
rows = soup.find_all(re.compile(r'Row$'))

parsed_data = []
for row in rows:
    row_dict = {}
    current_col = 0
    cells = row.find_all(re.compile(r'Cell$'))
    for cell in cells:
        # If the XML specifies a column index, jump to it (fixes the empty cell shifting bug)
        idx = cell.get('ss:Index')
        if idx:
            current_col = int(idx) - 1
        data_tag = cell.find(re.compile(r'Data$'))
        if data_tag:
            row_dict[current_col] = data_tag.text.strip()
        current_col += 1
    parsed_data.append(row_dict)

# 1. Convert to DataFrame
raw_df = pd.DataFrame(parsed_data)
# 2. Sort columns numerically to ensure correct order before applying headers
raw_df = raw_df.reindex(sorted(raw_df.columns), axis=1)
# 3. Apply Headers (Row index 5 is Row 6 in Excel)
sales_df = raw_df.iloc[6:].copy()
sales_df.columns = raw_df.iloc[5]
# 4. Clean up column names (removes hidden spaces)
sales_df.columns = [str(col).strip() for col in sales_df.columns]
# 5. Filter and Rename
sales_df = sales_df[['Item', 'Qty. Sold', 'Sales (USD)', 'Month']]
sales_df = sales_df.rename(columns={'Qty. Sold': 'Sales Units', 'Sales (USD)': 'Sales Amount'})
# 6. Convert to numeric and filter
sales_df['Sales Units'] = pd.to_numeric(sales_df['Sales Units'], errors='coerce')
sales_df = sales_df[sales_df["Sales Units"] >= 0]
# Check the unique months!
print("\nUnique Months Found:")
print(sales_df['Month'].unique())
# Remove the last row
sales_df = sales_df.iloc[:-1]
# Display to verify
display(sales_df)

Parsing XML... 

Unique Months Found:
['2025-02' '2025-01' '2025-03' nan]


,Item,Sales Units,Sales Amount,Month
6,COC1-PS3-US-VG-DLC,16.0,11.52,2025-02
7,COC1-PS3-US-VG-DLC,24.0,16.56,2025-01
8,COC1-PS3-US-VG-PSN,9.0,6.27,2025-02
9,COC1-PS3-US-VG-PSN,9.0,6.21,2025-03
10,COC1-PS3-US-VG-PSN,24.0,16.64,2025-01
...,...,...,...,...
3360,FAL2-XB1-US-VG-XBL,0.0,0.0,2025-01
3361,FAL2-XB1-US-VG-XBL,45.0,366.39,2025-02
3362,FAL2-XB1-US-VG-XBL,12.0,120.84,2025-03
3363,Setup Fee,1.0,125.0,2025-01


# Coop

In [17]:
file_path = 'DataSource/New Input/CustomGeneralLedgerwithAccountName_Division10_2025Jan2025Mar.xls'

print("Parsing XML General Ledger... please wait...")
with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
    soup = BeautifulSoup(f.read(), 'xml')
rows = soup.find_all(re.compile(r'Row$'))

parsed_data = []
for row in rows:
    row_dict = {}
    current_col = 0
    cells = row.find_all(re.compile(r'Cell$'))
    for cell in cells:
        # Prevent empty cells from shifting data to the left
        idx = cell.get('ss:Index')
        if idx:
            current_col = int(idx) - 1
        data_tag = cell.find(re.compile(r'Data$'))
        if data_tag:
            row_dict[current_col] = data_tag.text.strip()
        current_col += 1
    parsed_data.append(row_dict)

# 1. Convert to DataFrame and sort columns
raw_df = pd.DataFrame(parsed_data)
raw_df = raw_df.reindex(sorted(raw_df.columns), axis=1)
# 2. Dynamically find the header row (looks for 'Account' in the first column)
header_idx = raw_df[raw_df[0].astype(str).str.strip() == 'Account'].index[0]
# 3. Slice the dataframe to drop the top titles and set the headers
custom_df = raw_df.iloc[header_idx + 1:].copy()
custom_df.columns = raw_df.iloc[header_idx]
# 4. Clean up column names (removes hidden spaces)
custom_df.columns = [str(col).strip() for col in custom_df.columns]
# Display the clean table
title_process(custom_df)


Parsing XML General Ledger... please wait...


,Account,Type,Date,Document Number,Name,Memo,Debit,Credit,Balance,Account: Name,Division: Name,Game Title: Name,Game Title,Item
6,Cash and Cash Equivalents,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
7,Cash and Cash Equivalents:BOW Checking (old),NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
8,Cash and Cash Equivalents:BOW Checking-ACH Deb...,Check,2025-01-02T00:00:00.000,5418,BOW,Line of credit fee,NaN,500.0,-500.0,BOW Checking-ACH Debits only (xxxx3945),- No Division -,- No Game Title -,- No Game Title -,- No Item -
9,Cash and Cash Equivalents:BOW Checking-ACH Deb...,Customer Refund,2025-01-06T00:00:00.000,5430,Jeremy Coy,Backorder (85P53086V4553500D),NaN,121.25,-621.25,BOW Checking-ACH Debits only (xxxx3945),- No Division -,- No Game Title -,- No Game Title -,- No Item -
10,Cash and Cash Equivalents:BOW Checking-ACH Deb...,Check,2025-01-09T00:00:00.000,5458,Georgia Department of Revenue,NaN,NaN,100.91,-722.16,BOW Checking-ACH Debits only (xxxx3945),- No Division -,- No Game Title -,- No Game Title -,- No Item -
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100003,Sales:Sales-North America:Sales-Online Store:S...,Invoice,2025-02-10T00:00:00.000,NISINV-79398,Brett Schwartz,NaN,0.0,NaN,-5282631.480002659,Accounts Receivable: Online Store,NA Games-STORE,- No Game Title -,- No Game Title -,Free USPS Shipping (Priority Mail)
100004,Sales:Sales-North America:Sales-Online Store:S...,Invoice,2025-02-10T00:00:00.000,NISINV-80197,Imran Sher-Khan,NaN,0.0,NaN,-5282631.480002659,Accounts Receivable: Online Store,NA Games-STORE,- No Game Title -,- No Game Title -,Free USPS Shipping (Priority Mail)
100005,Sales:Sales-North America:Sales-Online Store:S...,Invoice,2025-02-10T00:00:00.000,NISINV-78783,Kevin Street,NaN,0.0,NaN,-5282631.480002659,Accounts Receivable: Online Store,NA Games-STORE,- No Game Title -,- No Game Title -,Free USPS Shipping (Priority Mail)
100006,Sales:Sales-North America:Sales-Online Store:S...,Invoice,2025-02-10T00:00:00.000,NISINV-79512,Sheridan Maxey,NaN,NaN,10.61,-5282642.09000266,Accounts Receivable: Online Store,NA Games-STORE,- No Game Title -,- No Game Title -,USPS Priority Mail® (Taxed)


In [21]:
coop_df = custom_df[custom_df["Item"] == "CO-OP AD Expense"].copy()
coop_df = coop_df[['Game Title', 'Debit', 'Credit', 'Date']]
coop_df['Debit'] = pd.to_numeric(coop_df['Debit'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False), errors='coerce').fillna(0)
coop_df['Credit'] = pd.to_numeric(coop_df['Credit'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False), errors='coerce').fillna(0)
coop_df["Coop_expense"] = coop_df["Debit"] + coop_df["Credit"]
coop_df['Date'] = pd.to_datetime(coop_df['Date'], errors='coerce')
coop_df['Month'] = coop_df['Date'].dt.strftime('%Y-%m')
coop_df = coop_df[['Game Title', 'Coop_expense', 'Month']]

display(coop_df)

,Game Title,Coop_expense,Month
49855,DISGAEA 7 NSW,35.18,2025-01
49856,TRAILS THROUGH DAYBREAK NSW,330.48,2025-01
49862,TRAILS THROUGH DAYBREAK 2 PS5,13023.36,2025-02
49863,PHANTOM BRAVE 2 NSW,12288.96,2025-02
49864,TRAILS THROUGH DAYBREAK PS5,844.56,2025-02
49865,PHANTOM BRAVE 2 PS5,10379.52,2025-02
49866,PHANTOM BRAVE 2 PS4,1058.76,2025-02
49867,TRAILS THROUGH DAYBREAK 2 NSW,10342.80,2025-02
49868,DISGAEA 7 PS5,170.04,2025-02
49869,TRAILS THROUGH DAYBREAK 2 PS4,1530.00,2025-02


# Paid PP

In [23]:
pp_df = custom_df[custom_df["Item"].notna() & custom_df["Item"].astype(str).str.startswith("Mark Down-")].copy()
pp_df = pp_df[['Game Title', 'Debit', 'Credit', 'Date']]
pp_df['Debit'] = pd.to_numeric(pp_df['Debit'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False), errors='coerce').fillna(0)
pp_df['Credit'] = pd.to_numeric(pp_df['Credit'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False), errors='coerce').fillna(0)
pp_df["Paid_PP"] = pp_df["Debit"] + pp_df["Credit"]
pp_df['Date'] = pd.to_datetime(pp_df['Date'], errors='coerce')
pp_df['Month'] = pp_df['Date'].dt.strftime('%Y-%m')
pp_df = pp_df[['Game Title', 'Paid_PP', 'Month']]

display(pp_df)

,Game Title,Paid_PP,Month
69695,VOID TERRARIUM 2 PS4,174.67,2025-02
69697,TRAILS THROUGH DAYBREAK NSW,4955.84,2025-02
69698,REYNATIS PS4,5117.44,2025-02
69699,TRAILS THROUGH DAYBREAK PS5,26.93,2025-02
69700,LEGEND OF LEGACY HD PS4,5071.00,2025-02
...,...,...,...
69824,TRAILS THROUGH DAYBREAK 2 NSW,367.20,2025-03
69825,RPG MAKER WITH PS5,1243.13,2025-03
69826,RPG MAKER WITH PS4,234.08,2025-03
69827,TRAILS THROUGH DAYBREAK 2 PS5,367.20,2025-03


# COGS

In [24]:
cogs_df = custom_df[custom_df["Account: Name"].astype(str).str.contains('Material Costs', na=False)].copy()
cogs_df = cogs_df[['Game Title', 'Debit', 'Credit', 'Date']]
cogs_df['Debit'] = pd.to_numeric(cogs_df['Debit'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False), errors='coerce').fillna(0)
cogs_df['Credit'] = pd.to_numeric(cogs_df['Credit'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False), errors='coerce').fillna(0)
cogs_df["Material_COGS"] = cogs_df["Debit"] + cogs_df["Credit"]
cogs_df['Date'] = pd.to_datetime(cogs_df['Date'], errors='coerce')
cogs_df['Month'] = cogs_df['Date'].dt.strftime('%Y-%m')
cogs_df = cogs_df[['Game Title', 'Material_COGS', 'Month']]

display(cogs_df)

,Game Title,Material_COGS,Month
29051,TRAILS THROUGH DAYBREAK PS4,9.62,2025-01
30476,PHANTOM BRAVE 2,12.25,2025-01
30480,TRAILS THROUGH DAYBREAK 2,52.81,2025-01
30491,PHANTOM BRAVE 2,20.45,2025-01
30492,PHANTOM BRAVE 2,20.79,2025-01
30493,KIMI NI TODOKE 2,2254.82,2025-01
30521,TRAILS THROUGH DAYBREAK 2,3.22,2025-01
30549,KIMI NI TODOKE 3,190.00,2025-01
30551,TRAILS THROUGH DAYBREAK 2,20.36,2025-01
30553,03-2024,22.49,2025-01


# Royalty

In [26]:
royalty_df = pd.read_excel(input_folder_path + "Royalty/Mar 2025 NISJ Royalty from Netsuite.xlsx", skiprows=4)
royalty_df = royalty_df[['Memo', 'Amount']]
royalty_df = royalty_df.rename(columns = {'Memo': 'Item'})
royalty_df = royalty_df.rename(columns = {'Amount': 'Royalty'})

royalty_df

,Item,Royalty
0,ARC1-PS3-US-VG-PSN,20.53
1,CDU3-PS4-US-PSN,124.88
2,CGI2-PSV-US-VG-PSN,20.99
3,CGS2-PSV-US-VG-DLC,13.74
4,CGS2-PSV-US-VG-PSN,20.99
...,...,...
192,YSD1-NSW-US-eshop,3451.23
193,YSD1-NSW-US-eshop-DLC,112.60
194,ZHP1-PSP-US-VG-DLC,0.34
195,ZHP1-PSP-US-VG-PSN,3.50


In [30]:
import os
royalty_df_list = []

for royalty_name in os.listdir(input_folder_path + "Royalty"):
    month, year, file_type = royalty_name.split(' ', 2)
    Month = f"{month} {year}"

    if file_type == "NISJ EUR Royalty from Netsuite.xlsx":
        royalty_df = pd.read_excel(input_folder_path + "Royalty/" + royalty_name, skiprows=0)
        royalty_df = royalty_df[['Memo', 'Amount (Debit)', 'Amount (Credit)']]
        royalty_df['Amount'] = royalty_df['Amount (Debit)'].fillna(0) + royalty_df['Amount (Credit)'].fillna(0)
    else:
        royalty_df = pd.read_excel(input_folder_path + "Royalty/" + royalty_name, skiprows=4)


    royalty_df = royalty_df[['Memo', 'Amount']]
    royalty_df = royalty_df.rename(columns = {'Memo': 'Item'})
    royalty_df = royalty_df.rename(columns = {'Amount': 'Royalty'})
    royalty_df['Month'] = pd.to_datetime(Month, format='%b %Y').strftime('%Y-%m')
    royalty_df_list.append(royalty_df)

royalty_df = pd.concat(royalty_df_list, ignore_index=True)
royalty_df

,Item,Royalty,Month
0,NaN,13059.69,2025-02
1,CDU3-PS4-EU-VG-PSNE,41.19,2025-02
2,CGI2-PSV-EU-VG-PSNE,23.29,2025-02
3,CGS2-PSV-EU-VG-DLC,8.14,2025-02
4,CGS2-PSV-EU-VG-PSNE,11.95,2025-02
...,...,...,...
1246,PBR2-PS5-US PSN Amourtize,15649.06,2025-03
1247,PBR2-NSW-EU-eshop Amourtize,5321.94,2025-03
1248,PBR2-PS4-EU PSN Amourtize,105.18,2025-03
1249,PBR2-PS5-EU PSN Amourtize,5198.98,2025-03


# Contact

In [33]:
contract_df = pd.read_csv(input_folder_path + "Game Title with Contract Name.csv")
contract_df = contract_df.rename(columns={'game_title': 'Game Title', 'contract_name': 'Contract'})
contract_df = contract_df[['Game Title', 'Contract']]
contract_df = title_process(contract_df)
contract_df

,Game Title,Contract
0,GIRAFFE & ANNIKA NSW,Giraffe and Annika
1,GIRAFFE & ANNIKA PS4,Giraffe and Annika
2,NEO ATLAS 1469 NSW,Giraffe and Annika
3,LA-MULANA 1 NSW,LA-MULANA 1&2
4,LA-MULANA 1 PS4,LA-MULANA 1&2
...,...,...
346,DENPA ONNA TO SEISHUN OTOKO,TBS
347,ECCENTRIC FAMILY,YTE
348,HANASAKU IROHA 1,YTE
349,HANASAKU IROHA 2,YTE


# Item Master List Merging

In [34]:
master_df = pd.read_excel(input_folder_path + "Item Master List.xlsx")

In [36]:
master_df

,Inactive,Internal ID,Name,Display Name,SubType,Description,Game Title,Base Price,Taxable,Amazon Item ID,...,Expense/COGS Account,Division,Date Created,Type,Game Type,Platform,MSRP,Royalty % of Sales,Royalty Per Unit Amount,Licensor
0,No,9800,YSN1-PS5-EU-VGN-DX-DE,Ys X: Nordics Deluxe Edition (PS5),NaN,Ys X: Nordics Deluxe Edition (PS5),03-2025 : Ys 10 : Ys 10 PS5,NaN,Yes,NaN,...,COGS-Europe:Wholesale_Material Cost,EU Games,2025-05-01 09:39:00,Inventory Item,Physical,PS5,59.99 EUR,NaN,0.0,Nihon Falcom
1,No,9799,YSN1-PS5-EU-VGN-DX,Ys X: Nordics Deluxe Edition (PS5),NaN,Ys X: Nordics Deluxe Edition (PS5),03-2025 : Ys 10 : Ys 10 PS5,NaN,Yes,NaN,...,COGS-Europe:Wholesale_Material Cost,EU Games,2025-05-01 09:39:00,Inventory Item,Physical,PS5,59.99 EUR,NaN,0.0,Nihon Falcom
2,No,9798,YSM1-PS5-EU-VGN-DX-DE,NaN,NaN,Ys IX: Monstrum Nox (PS5) Deluxe Edition,03-2024 : Ys IX Monstrum Nox : Ys IX Monstrum ...,49.99,Yes,NaN,...,COGS-Europe:Wholesale_Material Cost,EU Games,2025-05-01 09:34:00,Inventory Item,Physical,PS5,49.99,NaN,0.0,Nihon Falcom
3,No,9797,YSM1-PS5-EU-VGN-DX,NaN,NaN,Ys IX: Monstrum Nox (PS5) Deluxe Edition,03-2024 : Ys IX Monstrum Nox : Ys IX Monstrum ...,49.99,Yes,NaN,...,COGS-Europe:Wholesale_Material Cost,EU Games,2025-05-01 09:33:00,Inventory Item,Physical,PS5,49.99,NaN,0.0,Nihon Falcom
4,No,9796,TBH1-NSW-EU-CART,NaN,NaN,The Legend of Heroes: Trails beyond the Horizo...,03-2026 : Trails beyond the Horizon : Trails b...,0.00,Yes,NaN,...,COGS-Europe:Wholesale_Material Cost,EU Games,2025-04-21 16:18:00,Inventory Item,Components,NSW,0.00,NaN,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9161,Yes,16,BAD1-PSP-IT-VG,BAD1-PSP-IT-VG,NaN,Badman PSP-Italian,My Lord!? : My Lord!? 1 PSP-EU,NaN,No,NaN,...,COGS-Europe:Wholesale_Material Cost,EU Games,2009-10-11 17:39:00,Inventory Item,NaN,NaN,NaN,NaN,3.5,SCEI-EUR
9162,Yes,15,BAD1-PSP-DE-VG,BAD1-PSP-DE-VG,NaN,Badman PSP-German,My Lord!? : My Lord!? 1 PSP-EU,NaN,No,NaN,...,COGS-Europe:Wholesale_Material Cost,EU Games,2009-10-11 17:39:00,Inventory Item,NaN,NaN,NaN,NaN,3.5,SCEI-EUR
9163,Yes,14,BAD1-PSP-FR-VG,BAD1-PSP-FR-VG,NaN,Badman PSP-French,My Lord!? : My Lord!? 1 PSP-EU,NaN,No,NaN,...,COGS-Europe:Wholesale_Material Cost,EU Games,2009-10-11 17:39:00,Inventory Item,NaN,NaN,NaN,NaN,3.5,SCEI-EUR
9164,Yes,13,BAD1-PSP-AU-VG,BAD1-PSP-AU-VG,NaN,Badman PSP-Australian,My Lord!? : My Lord!? 1 PSP-EU,NaN,No,NaN,...,COGS-Europe:Wholesale_Material Cost,EU Games,2009-10-11 17:39:00,Inventory Item,NaN,NaN,NaN,NaN,3.5,SCEI-EUR


In [37]:
marginal_profit_df = master_df[["Inactive", "Game Title", "Name", "Platform", "Division", 'Game Type']].copy()
marginal_profit_df = marginal_profit_df[marginal_profit_df["Inactive"] == "No" ]
marginal_profit_df["Title"] = master_df["Game Title"]
marginal_profit_df = marginal_profit_df[["Title", "Game Title", "Name",  "Platform", "Division", 'Game Type']]
marginal_profit_df = marginal_profit_df.rename(columns = {'Name': 'Item'})
title_process(marginal_profit_df)

marginal_profit_df = marginal_profit_df.merge(sales_df, on = "Item", how = "left")
marginal_profit_df = marginal_profit_df.merge(coop_df, on = ["Game Title","Month"], how = "left")
marginal_profit_df = marginal_profit_df.merge(pp_df, on = ["Game Title","Month"], how = "left")
marginal_profit_df = marginal_profit_df.merge(cogs_df, on = ["Game Title","Month"], how = "left")
marginal_profit_df = marginal_profit_df.merge(royalty_df, on = ["Item","Month"], how = "left")
marginal_profit_df = marginal_profit_df.merge(contract_df, on = ["Game Title"], how = "left")
# 1. List the columns that need to be math numbers
math_columns = ["Sales Amount", "Paid_PP", "Coop_expense", "Material_COGS", "Royalty"]

# 2. Clean them (remove $ and commas) and force them to be numeric
for col in math_columns:
    if col in marginal_profit_df.columns:
        marginal_profit_df[col] = pd.to_numeric(
            marginal_profit_df[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False),
            errors='coerce'
        ).fillna(0)

# 3. Now do the math! (We don't need .fillna(0) here anymore because the loop above handled it)
marginal_profit_df["Marginal Profit"] = (
    marginal_profit_df["Sales Amount"] -
    marginal_profit_df["Paid_PP"] -
    marginal_profit_df["Coop_expense"] -
    marginal_profit_df["Material_COGS"] -
    marginal_profit_df["Royalty"]
)

# 4. Finish your dataframe formatting
marginal_profit_df = marginal_profit_df.drop(columns = ["Game Title"])
marginal_profit_df = marginal_profit_df.assign(**{'Game Type': marginal_profit_df.pop('Game Type')})
marginal_profit_df = marginal_profit_df.assign(**{'Contract': marginal_profit_df.pop('Contract')})

display(marginal_profit_df)

,Title,Item,Platform,Division,Sales Units,Sales Amount,Month,Coop_expense,Paid_PP,Material_COGS,Royalty,Marginal Profit,Game Type,Contract
0,03-2025 : Ys 10 : Ys 10 PS5,YSN1-PS5-EU-VGN-DX-DE,PS5,EU Games,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,Physical,Ys 10
1,03-2025 : Ys 10 : Ys 10 PS5,YSN1-PS5-EU-VGN-DX,PS5,EU Games,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,Physical,Ys 10
2,03-2024 : Ys IX Monstrum Nox : Ys IX Monstrum ...,YSM1-PS5-EU-VGN-DX-DE,PS5,EU Games,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,Physical,Ys 9 PS5
3,03-2024 : Ys IX Monstrum Nox : Ys IX Monstrum ...,YSM1-PS5-EU-VGN-DX,PS5,EU Games,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,Physical,Ys 9 PS5
4,03-2026 : Trails beyond the Horizon : Trails b...,TBH1-NSW-EU-CART,NSW,EU Games,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,Components,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13871,Goods,W03PR-GP001,NaN,NA Games : NA Games-STORE,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,NaN,Ys Anniversary
13872,Goods,W03PR-GP001,NaN,NA Games : NA Games-STORE,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,NaN,Trails through Daybreak - Goods
13873,Goods,W03PR-GP001,NaN,NA Games : NA Games-STORE,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,NaN,Ys 10 - Goods
13874,Goods,W03PR-GP001,NaN,NA Games : NA Games-STORE,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,NaN,Trails through Daybreak 2 - Goods


Export to csv

In [40]:
marginal_profit_df.to_csv(output_folder_path + 'marginal_profit_report.csv')